# Autoencoder Training

Train an LSTM autoencoder on fault-free satellite telemetry for anomaly detection.

## Prerequisites

```bash
pip install -e ".[notebooks]"
```

Place your data under `data/processed/merged_data/simulations_year/` (see `docs/data_format.md`).

## Setup and imports

In [ ]:
import torch

from sat_anomaly.config import get_lstm_config, validate_config
from sat_anomaly.models.autoencoder import create_lstm_autoencoder, create_rnn_autoencoder
from sat_anomaly.models.training import train_autoencoder
from sat_anomaly.visualization.plots import plot_training_history
from sat_anomaly.data.loader import load_fault_free_data, create_data_loaders
from sat_anomaly.data.preprocessor import normalize_features, create_grouped_time_windows, split_sequences

print("Import successful")

## Build config

In [ ]:
config = get_lstm_config()
config['epochs'] = 100
assert validate_config(config)
config

## Load and preprocess data

In [ ]:
data = load_fault_free_data(config['data_path'])
normalized_data, _scaler = normalize_features(data)

_numeric_cols = normalized_data.select_dtypes(include=['number']).columns
feature_names = [c for c in _numeric_cols if c not in ['time_ns', 'time_s', 'label_any_fault']]
config['feature_names'] = feature_names

windows = create_grouped_time_windows(
    normalized_data,
    group_cols=['sequence_id'],
    window_size=config['window_size'],
    step_size=config['step_size']
)
train_data, val_data = split_sequences(windows, train_ratio=config['train_ratio'])
train_loader, val_loader = create_data_loaders(train_data, val_data, config['batch_size'])

## Create model

In [ ]:
model_type = config['model_type']
if model_type == 'lstm_ae':
    model = create_lstm_autoencoder(config)
elif model_type == 'rnn_ae':
    model = create_rnn_autoencoder(config)
else:
    raise ValueError(f"Unsupported model type: {model_type}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f'Using device: {device}')

## Train

In [ ]:
train_results = train_autoencoder(model, train_loader, val_loader, config)
print('Training completed!')
plot_training_history(train_results['train_losses'], train_results['val_losses'])

## Visualize reconstruction

In [ ]:
import random
import math
import matplotlib.pyplot as plt

model.eval()
batch = next(iter(val_loader))
inputs = batch if isinstance(batch, torch.Tensor) else batch[0]
idx = random.randrange(inputs.size(0))
seq = inputs[idx:idx+1].to(device)

with torch.no_grad():
    recon = model(seq)

seq_np = seq.squeeze(0).cpu().numpy()
recon_np = recon.squeeze(0).cpu().numpy()
T, F = seq_np.shape

ncols = 4
nrows = math.ceil(F / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 2.2 * nrows), sharex=True)
axes = axes.ravel()

for fi in range(F):
    ax = axes[fi]
    name = feature_names[fi] if fi < len(feature_names) else f'Feature {fi}'
    ax.plot(range(T), seq_np[:, fi], label='Original', color='C0', linewidth=1.0)
    ax.plot(range(T), recon_np[:, fi], label='Reconstructed', color='C1', linestyle='--', linewidth=1.0)
    ax.set_title(name, fontsize=9)
    ax.grid(True, alpha=0.3)

for j in range(F, len(axes)):
    axes[j].axis('off')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2)
plt.suptitle('Validation sample reconstruction (all features)')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()